# OmniShift - Multiply-Free CNN Training

| `METHOD` | Conv layer | BN | Activation | Paper |
|---|---|---|---|---|
| `fp32` | `nn.Conv2d` | standard | ReLU | baseline |
| `deepshift` | `ShiftConv2d` W in {+/-2^p} | standard | ReLU | CVPR-W 2021, arXiv 1905.13298 |
| `apot` | `APoTConv2d` additive PoT grid | standard | ReLU | ICLR 2020, arXiv 1909.13144 |
| `denseshift` | `DenseShiftConv2d` sign*2^p | standard | ReLU | ICCV 2023, arXiv 2208.09708 |
| `s3shift` | `S3ShiftConv2d` sign*gate*2^p | standard | ReLU | NeurIPS 2021, arXiv 2107.03453 |
| `fogzo` | `FogzoShiftConv2d` ±2^p+ZO-STE | standard | ReLU | NeurIPS 2025, arXiv 2510.23926 |
| `aptq` | `APTQTernaryConv2d` {0,+/-2^p} | standard | ReLU | Sensors 2024, DOI 10.3390/s24010181 |
| `omnishift` | `SparseShiftConv2dEWGS` W in {0,+/-2^p} | `PoTBatchNorm2dEWGS` | `PoTActivationEWGS` | this work |

## 1 . Setup

In [ ]:
import os, subprocess, sys

REPO = "/kaggle/working/OmniShift"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "https://github.com/CryAndRRich/OmniShift.git", REPO],
        check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!pip install -q pyyaml
print("Setup done. CWD:", os.getcwd())

---
## 2 . Config  -- edit this cell

In [ ]:
METHOD = "omnishift"
BACKBONE = "resnet20"
DATASET = "cifar10"
SEED = 42
EPOCHS = 200

---
## 3 . Train

In [ ]:
import shutil
import torch
print(f'Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f"PyTorch: {torch.__version__}")

from scripts.run_experiment import build_cfg, run

# Back up the pre-stabilizer log once, so the old (unstable) curve
# survives for the before/after comparison plot below.
tag = f"{METHOD}_{BACKBONE}_{DATASET}_seed{SEED}"
old_log = f"logs/{BACKBONE}/{tag}.json"
backup_log = f"logs/{BACKBONE}/{tag}_prestab.json"
if os.path.exists(old_log) and not os.path.exists(backup_log):
    shutil.copy(old_log, backup_log)
    print(f"Backed up old log -> {backup_log}")

cfg = build_cfg(METHOD, BACKBONE, DATASET, seed=SEED, epochs=EPOCHS)
result = run(cfg)

print(f'\ntest_acc = {result["test_acc"]:.4f}')
print(f'energy = {result["final_ops"]["energy_GpJ"]:.4f} GpJ')
print(f'sparsity = {result["final_sparsity"]:.2%}')
print(f'params = {result["n_params"] / 1e6:.3f} M')

---
## 4 . Results

In [ ]:
import json
import matplotlib.pyplot as plt

run_name = result["run_name"]
log_path = f"logs/{BACKBONE}/{run_name}_{DATASET}_seed{SEED}.json"
with open(log_path) as f:
    data = json.load(f)

meta = data["meta"]
log = data["log"]
ep = [e["epoch"] + 1 for e in log]
tr_loss = [e["tr_loss"] for e in log]
val_loss = [e["val_loss"] for e in log]
tr_acc = [e["tr_acc"] for e in log]
val_acc = [e["val_acc"] for e in log]
flip = [e.get("flip_rate", 0.0) for e in log]
frozen = [e.get("frozen_frac", 0.0) for e in log]

# old (pre-stabilizer) curve, if the backup exists
old_val_acc = None
if os.path.exists(backup_log):
    with open(backup_log) as f:
        old = json.load(f)["log"]
    old_val_acc = [e["val_acc"] for e in old]

has_flip = any(v > 0 for v in flip) or any(v > 0 for v in frozen)
n_panels = 3 if has_flip else 2
fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 5))
fig.suptitle(f"{run_name}  |  {BACKBONE}  |  {DATASET}", fontsize=13)

ax1, ax2 = axes[0], axes[1]
ax1.plot(ep, tr_loss, label="Train", linewidth=1.5)
ax1.plot(ep, val_loss, label="Val", linewidth=1.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

if old_val_acc is not None:
    ax2.plot(range(1, len(old_val_acc) + 1), old_val_acc,
             label="Val (pre-stabilizer)", linewidth=1.0, alpha=0.45, color="gray")
ax2.plot(ep, tr_acc, label="Train", linewidth=1.5)
ax2.plot(ep, val_acc, label="Val", linewidth=1.5)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

if has_flip:
    ax3 = axes[2]
    ax3.plot(ep, flip, label="Flip rate (EMA)", linewidth=1.5)
    ax3.set_xlabel("Epoch")
    ax3.set_ylabel("Flip rate")
    ax3.set_title("Oscillation: flip rate & frozen fraction")
    ax3.grid(True, alpha=0.3)
    ax3b = ax3.twinx()
    ax3b.plot(ep, frozen, label="Frozen frac", linewidth=1.5, color="tab:orange")
    ax3b.set_ylabel("Frozen fraction")
    lines1, labels1 = ax3.get_legend_handles_labels()
    lines2, labels2 = ax3b.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labels1 + labels2, loc="center right")

plt.tight_layout()
plt.savefig(f"{run_name}_{DATASET}_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# stability metric for the paper: mean |delta val_acc| per epoch after BN warmup
def mean_delta(vals, start=31):
    ds = [abs(vals[i] - vals[i - 1]) for i in range(start, len(vals))]
    return sum(ds) / len(ds) if ds else 0.0

print(f'\n{"="*50}')
print(f'  Method    : {meta["method"]}')
print(f'  Backbone  : {meta["backbone"]}')
print(f'  Dataset   : {meta["dataset_name"]}')
print(f'  Test Acc  : {meta["test_acc"]:.4f}')
print(f'  Best Val  : {meta["best_val"]:.4f}  (epoch {meta["best_epoch"] + 1})')
print(f'  Sparsity  : {meta["final_sparsity"]:.2%}')
print(f'  Energy    : {meta["final_ops"]["energy_GpJ"]:.4f} GpJ')
print(f'  Params    : {meta["n_params"] / 1e6:.3f} M')
print(f'  mean|dVal|: {mean_delta(val_acc) * 100:.2f} pp/epoch (ep>30)')
if old_val_acc is not None:
    print(f'  (pre-stab): {mean_delta(old_val_acc) * 100:.2f} pp/epoch')
print(f'{"="*50}')